# Filter balance (u, g, r, i, z, y) versus time and survey type, in baseline survey v5.3

This notebook complements `04_DDF_NVisitsVsTime`, `07_WFDMiniMicroS_NVisitsVsTime` (number of visits versus time) and `05_DDF_DepthVsTime`, `09_WFDLowDust_DepthVsTime` (depth versus time). It checks the **distribution of the visits among the six filters**, **month by month over the 10 years** of the survey, separately for each survey type: **WFD**, **DDF**, **mini-surveys** and **micro-surveys**.

- creation date: 2026-10-11
- companion notebooks: `02_SurveyAreas` (visits per region over 10 years), `04_DDF_NVisitsVsTime`, `07_WFDMiniMicroS_NVisitsVsTime`, `05_DDF_DepthVsTime`, `09_WFDLowDust_DepthVsTime`
- survey modes definition: https://github.com/lsst/survey-strategy_lsst_io

## Goal, hypothesis and outputs

**Hypothesis to test**: the filter balance should be **different from one survey type to another** (the WFD, the DDFs and the mini-surveys do not have the same science needs), but **stable in time** for a given survey type.

**Quantity studied**: for a given survey category and a given month, the fraction of visits in filter `b` is

`f_b(month) = N_b(month) / sum_b N_b(month)`

The time granularity is the **month** (1/12 year, 120 months over 10 years). The month is a good choice because it is close to the lunar cycle (29.5 days): the choice of the filter depends on the Moon phase (`u` is observed mostly around new Moon, `y` and `z` rather around full Moon), so a bin shorter than the lunar cycle would show a lunar modulation that has nothing to do with a drift of the strategy.

**Categories** (each visit is counted once, as in `07_WFDMiniMicroS_NVisitsVsTime`):

| Category | Definition |
|---|---|
| `WFD` | pixels of the scheduler map: low dust (+ euclid overlap), bulge, Virgo, SMC-LMC |
| `WFD low dust` | subset of the WFD (the region used for cosmology), **included in** `WFD` |
| `DDF` | sum of all the DDFs (`scheduler_note`) |
| mini-surveys | `NES`, `SCP`, `MW non-WFD`, and their total |
| micro-surveys without DDF | `RGES`, `TwilightNEO`, `ToO`, and their total |

Categories with too few visits are removed automatically (threshold `MIN_VISITS_TOTAL`), and months with too few visits in a category are masked (threshold `MIN_VISITS_MONTH`), because a fraction has no statistical meaning with a handful of visits.

**Steps**

1. Region labels (scheduler map) and loading of the visits.
2. Number of visits per month, per filter and per category (`.csv` table).
3. Filter balance over the 10 years, per category (table and Figures 3 and 4).
4. Monthly fractions (Figures 1 and 2).
5. Time stability test: deviation from the statistical fluctuation, linear drift, chi-square test of homogeneity (Figure 5 and tables).

**Input**: the OpSim database `baseline_v5.3.6_10yrs.db` (variable `opsdb`).

**Outputs**: the tables (`.csv`) go to `DATA10_FilterBalance/`; all figures (`.png` and `.pdf`) go to `FIG10_FilterBalance/`.

## 0. Setup

In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt
from scipy import stats

from rubin_scheduler.utils import SURVEY_START_MJD
from rubin_scheduler.scheduler.utils import CurrentAreaMap

In [ ]:
plt.rcParams.update(
    {
        "font.size": 14,  # general size
        "axes.labelsize": 16,  # X and Y labels
        "axes.titlesize": 16,  # axes title
        "xtick.labelsize": 14,  # X ticks
        "ytick.labelsize": 14,  # Y ticks
        "legend.fontsize": 14,  # legend text
        "legend.title_fontsize": 14,
    }
)

In [ ]:
# Output directories: tables go to DATA_DIR, figures go to FIG_DIR
DATA_DIR = "DATA10_FilterBalance"
FIG_DIR = "FIG10_FilterBalance"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# OpSim database. Replace by another path to analyse another run.
opsdb = "/Users/dagoret/DATA/OpSim/baseline_v5.3.6_10yrs.db"
opsim_name = os.path.split(opsdb)[-1].replace(".db", "")
print(opsdb)

YEAR = 365.25  # days
N_YEARS = 10
N_MONTHS = 12 * N_YEARS  # time granularity: 1 month = 1/12 year
BANDS = list("ugrizy")
# Usual Rubin colours of the six filters
BAND_COLORS = {
    "u": "#0c71ff",
    "g": "#49be61",
    "r": "#c61c00",
    "i": "#ffc200",
    "z": "#f341a2",
    "y": "#5d0000",
}
NSIDE = 64  # HEALPix resolution of the scheduler map

# Statistical thresholds
MIN_VISITS_MONTH = 50  # below this, the fraction of the month is not computed
MIN_VISITS_TOTAL = 1000  # below this, the category is removed from the study
MIN_GOOD_MONTHS = 12  # minimum number of usable months to test the stability

# INDICATIVE reference filter balance of the WFD (target number of visits per field and per filter,
# values of the previous baseline): to be checked / adapted to v5.3 before drawing any conclusion.
REF_WFD_VISITS = {"u": 56, "g": 80, "r": 184, "i": 184, "z": 160, "y": 160}

# Region definition: name -> list of labels of the scheduler map.
# 'euclid_overlap' is merged with 'lowdust' (as in 02_SurveyAreas and 07).
WFD_REGIONS = {
    "low dust": ["lowdust", "euclid_overlap"],
    "bulge": ["bulgy"],
    "virgo": ["virgo"],
    "SMC-LMC": ["LMC_SMC"],
}
MINI_SURVEYS = {
    "NES": ["nes"],
    "SCP": ["scp"],
    "MW non-WFD": ["dusty_plane"],
}

## 1. Region labels and loading of the visits

`CurrentAreaMap` returns one survey-mode label per HEALPix pixel. A label present in the map but not listed in `WFD_REGIONS` or `MINI_SURVEYS` is automatically added as a mini-survey, so no region is silently forgotten.

All visits are read once and sorted in time. Time is expressed in **years since the survey start** (`SURVEY_START_MJD`).

- **Micro-surveys**: selected from `scheduler_note` (`ToO`, `twilight_near_sun`, `RGES`, `DD`), with the priority ToO > TwilightNEO > RGES > DDF.
- **Other visits**: label of the HEALPix pixel containing the visit centre.

In [ ]:
# Scheduler footprint: one survey-mode label per HEALPix pixel
skymap = CurrentAreaMap(nside=NSIDE)
maps, labels = skymap.return_maps()
labels = np.asarray(labels).astype(str)
label_names = [l for l in np.unique(labels) if l != ""]
print("Labels in the target map:", label_names)

# Labels configured above but absent from the map
for name, labs in {**WFD_REGIONS, **MINI_SURVEYS}.items():
    for l in labs:
        if l not in label_names:
            print(f"WARNING: label {l} of region {name} is not in the target map")

# Labels of the map not configured above: added as mini-surveys
used_labels = {l for labs in list(WFD_REGIONS.values()) + list(MINI_SURVEYS.values()) for l in labs}
for l in label_names:
    if l not in used_labels:
        print(f"Label {l} is not in WFD_REGIONS or MINI_SURVEYS: added as a mini-survey")
        MINI_SURVEYS[l] = [l]

mini_names = list(MINI_SURVEYS)
wfd_labels = [l for labs in WFD_REGIONS.values() for l in labs]
mini_labels = [l for labs in MINI_SURVEYS.values() for l in labs]

In [ ]:
query = (
    "select observationStartMJD, night, band, fieldRA, fieldDec, "
    "visitExposureTime, scheduler_note from observations"
)
conn = sqlite3.connect(opsdb)
visits = pd.read_sql(query, conn)
conn.close()

visits = visits.sort_values("observationStartMJD").reset_index(drop=True)
visits["years"] = (visits["observationStartMJD"] - SURVEY_START_MJD) / YEAR
N_ALL = len(visits)
print(f'{N_ALL} visits in the survey, duration {visits["years"].max():.3f} years')

# Micro-surveys from scheduler_note (one tag per visit)
note = visits["scheduler_note"].fillna("")
is_too = note.str.contains("ToO").values
is_twilight = note.str.contains("twilight_near_sun").values
is_rges = note.str.contains("RGES").values
is_ddf = note.str.contains("DD").values & ~is_rges
visits["micro"] = np.select(
    [is_too, is_twilight, is_rges, is_ddf],
    ["ToO", "TwilightNEO", "RGES", "DDF"],
    default="",
)
is_micro = (visits["micro"] != "").values

# Label of the HEALPix pixel containing the visit centre (RA, Dec in degrees)
pix = hp.ang2pix(NSIDE, visits["fieldRA"].values, visits["fieldDec"].values, lonlat=True)
visits["label"] = labels[pix]

# Index of the filter (0..5); -1 if the filter is not in BANDS
band_idx = visits["band"].map({b: i for i, b in enumerate(BANDS)}).fillna(-1).astype(int).values
valid_band = band_idx >= 0
print(f"Visits in a filter outside {BANDS}: {(~valid_band).sum()}")
print(visits["band"].value_counts().to_string())

**Boolean masks.** `masks` stores, for each category, a boolean array over the time-sorted visits. Check: WFD + DDF + mini-surveys + micro-surveys without DDF + unassigned visits must add up to the number of visits of the survey. `WFD low dust` is a subset of `WFD`: it is not part of this sum.

The list `CATEGORIES` is the display order of all the figures; categories with fewer than `MIN_VISITS_TOTAL` visits, or fewer than `MIN_GOOD_MONTHS` usable months, are removed from it (with a message).

In [ ]:
sky_label = visits["label"].values
not_micro = ~is_micro

masks = {}
masks["All visits"] = np.ones(N_ALL, dtype=bool)
masks["WFD"] = not_micro & np.isin(sky_label, wfd_labels)
masks["WFD low dust"] = not_micro & np.isin(sky_label, WFD_REGIONS["low dust"])
masks["DDF"] = visits["micro"].values == "DDF"
masks["Mini-surveys (total)"] = not_micro & np.isin(sky_label, mini_labels)
for name, labs in MINI_SURVEYS.items():
    masks[name] = not_micro & np.isin(sky_label, labs)
for name in ["RGES", "TwilightNEO", "ToO"]:
    masks[name] = visits["micro"].values == name
masks["Micro-surveys without DDF (total)"] = is_micro & ~masks["DDF"]
masks["Unassigned"] = not_micro & ~masks["WFD"] & ~masks["Mini-surveys (total)"]

n_sum = sum(
    masks[n].sum()
    for n in ["WFD", "DDF", "Mini-surveys (total)", "Micro-surveys without DDF (total)", "Unassigned"]
)
print(f"WFD + DDF + mini + micro without DDF + unassigned = {n_sum}, all visits = {N_ALL}")
assert n_sum == N_ALL, "a visit is counted twice or not at all"
print(f'Unassigned visits: {masks["Unassigned"].sum()}')

# Display order of the categories
CATEGORIES = (
    ["All visits", "WFD", "WFD low dust", "DDF", "Mini-surveys (total)"]
    + mini_names
    + ["RGES", "TwilightNEO", "ToO", "Micro-surveys without DDF (total)"]
)
CATEGORIES = [c for c in CATEGORIES if c in masks]

## 2. Number of visits per month, per filter and per category

The month of a visit is `floor(12 * time in years)`; the few visits beyond 10 years, if any, are put in the last month. `counts[category]` is an array `(120 months, 6 filters)`.

From these numbers:

- `f_mean`: mean fraction of each filter, computed on the usable months (at least `MIN_VISITS_MONTH` visits in the month);
- `frac`: monthly fraction `f_b(month)` (NaN for the months that are not usable);
- `sigma`: standard deviation **expected from binomial statistics alone** if the true fraction were constant and equal to `f_mean`, i.e. `sqrt(f_mean (1 - f_mean) / N_month)`. It is the minimum fluctuation that must be observed from one month to another.

The full table of the numbers of visits is saved in `DATA10_FilterBalance/` (`..._visits_per_month_band.csv`).

In [ ]:
t_all = visits["years"].values
month_idx = np.minimum(np.floor(t_all * 12).astype(int), N_MONTHS - 1)
x = (np.arange(N_MONTHS) + 0.5) / 12  # centre of each month, in years

counts = {}
for cat in CATEGORIES:
    c = np.zeros((N_MONTHS, len(BANDS)), dtype=int)
    m = masks[cat] & valid_band
    np.add.at(c, (month_idx[m], band_idx[m]), 1)
    counts[cat] = c

# Categories with too few visits are removed from the study
kept = []
for cat in CATEGORIES:
    n_good = int((counts[cat].sum(axis=1) >= MIN_VISITS_MONTH).sum())
    if counts[cat].sum() >= MIN_VISITS_TOTAL and n_good >= MIN_GOOD_MONTHS:
        kept.append(cat)
    else:
        print(f"Category removed (too few visits): {cat}, {counts[cat].sum()} visits, {n_good} usable months")
CATEGORIES = kept

n_month = {cat: counts[cat].sum(axis=1) for cat in CATEGORIES}
good = {cat: n_month[cat] >= MIN_VISITS_MONTH for cat in CATEGORIES}

f_mean, frac, sigma = {}, {}, {}
for cat in CATEGORIES:
    g = good[cat]
    n_safe = np.maximum(n_month[cat], 1)[:, None]
    fm = counts[cat][g].sum(axis=0) / counts[cat][g].sum()
    f_mean[cat] = fm
    frac[cat] = np.where(g[:, None], counts[cat] / n_safe, np.nan)
    sigma[cat] = np.where(g[:, None], np.sqrt(fm * (1 - fm) / n_safe), np.nan)

# Saving: one table in long format (category, month, number of visits per filter)
frames = []
for cat in CATEGORIES:
    df = pd.DataFrame(counts[cat], columns=BANDS)
    df.insert(0, "category", cat)
    df.insert(1, "month", np.arange(N_MONTHS) + 1)
    df.insert(2, "year_center", x)
    df["N_total"] = n_month[cat]
    frames.append(df)
pd.concat(frames).to_csv(os.path.join(DATA_DIR, f"{opsim_name}_visits_per_month_band.csv"), index=False)
print("Categories studied:", CATEGORIES)

## 3. Filter balance over the 10 years

For each category: total number of visits, number of visits per filter, and **fraction (%) of each filter**. This is the quantity to compare from one survey type to another. The table is saved in `DATA10_FilterBalance/` (`..._filter_balance_10yr.csv`).

**Colours.** The reference is the balance of the **whole survey** (row `All visits`). A cell (number `N_b` or fraction `f_b`) is coloured when the category deviates **significantly** from this reference: **red** = excess of the filter, **blue** = deficit, stronger colour for a larger deviation (saturating at 10 sigma). The significance is the binomial z-score `z = (N_b - N_cat f_all_b) / sqrt(N_cat f_all_b (1 - f_all_b))`, where `N_cat` is the number of visits of the category and `f_all_b` the fraction of filter `b` in the whole survey. A cell is coloured if `|z| > Z_THRESHOLD` (3 sigma) **and** the fraction differs by more than `MIN_DELTA_PP` (1 percentage point), because with hundreds of thousands of visits tiny deviations are statistically significant without being meaningful. The z-scores are saved in `..._filter_balance_zscores.csv`.

Caution: the visits are correlated (pairs, blobs in one filter), so the real significance is lower than the z-score suggests; `WFD` is by construction very close to `All visits`, since it contains most of the visits.

In [ ]:
rows = {}
for cat in CATEGORIES:
    n = counts[cat].sum(axis=0)
    row = {"N visits": int(n.sum())}
    for i, b in enumerate(BANDS):
        row[f"N_{b}"] = int(n[i])
    for i, b in enumerate(BANDS):
        row[f"f_{b} (%)"] = round(100 * n[i] / n.sum(), 2)
    rows[cat] = row
balance = pd.DataFrame(rows).T
for col in ["N visits"] + [f"N_{b}" for b in BANDS]:
    balance[col] = balance[col].astype(int)
balance.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_filter_balance_10yr.csv"))

# Significance of the deviation of each category from the balance of the whole survey ('All visits').
# If a category had the same balance as the whole survey, its number of visits N_b in filter b would be
# binomial with expectation N_cat * f_all_b, hence z = (N_b - N_cat f_all_b) / sqrt(N_cat f_all_b (1 - f_all_b)).
# The same z applies to the number N_b and to the fraction f_b of the category.
Z_THRESHOLD = 3.0  # significance threshold (sigma)
MIN_DELTA_PP = 1.0  # minimum deviation of the fraction (percentage points) for a cell to be coloured
f_all = counts["All visits"].sum(axis=0) / counts["All visits"].sum()
z_score = pd.DataFrame(np.nan, index=balance.index, columns=balance.columns)
delta_pp = pd.DataFrame(np.nan, index=balance.index, columns=balance.columns)
for cat in CATEGORIES:
    n_cat = counts[cat].sum()
    n_obs = counts[cat].sum(axis=0)
    expected = n_cat * f_all
    with np.errstate(divide="ignore", invalid="ignore"):
        z_b = (n_obs - expected) / np.sqrt(expected * (1 - f_all))
    d_b = 100 * (n_obs / n_cat - f_all)  # deviation of the fraction, in percentage points
    for i, b in enumerate(BANDS):
        for col in (f"N_{b}", f"f_{b} (%)"):
            z_score.loc[cat, col] = z_b[i]
            delta_pp.loc[cat, col] = d_b[i]
z_score.round(2).to_csv(os.path.join(DATA_DIR, f"{opsim_name}_filter_balance_zscores.csv"))


def colour_cells(_):
    # Red: excess of the filter with respect to the whole survey; blue: deficit.
    # The colour is stronger when the deviation is larger (saturates at 10 sigma).
    css = pd.DataFrame("", index=balance.index, columns=balance.columns)
    significant = (z_score.abs() > Z_THRESHOLD) & (delta_pp.abs() > MIN_DELTA_PP)
    for cat in balance.index:
        for col in balance.columns:
            if significant.loc[cat, col]:
                alpha = 0.25 + 0.55 * min(abs(z_score.loc[cat, col]) / 10, 1)
                rgb = "214, 39, 40" if z_score.loc[cat, col] > 0 else "31, 119, 180"
                css.loc[cat, col] = f"background-color: rgba({rgb}, {alpha:.2f})"
    return css


fmt = {"N visits": "{:,.0f}"}
fmt.update({f"N_{b}": "{:,.0f}" for b in BANDS})
fmt.update({f"f_{b} (%)": "{:.2f}" for b in BANDS})
balance.style.apply(colour_cells, axis=None).format(fmt)

In [ ]:
def save_fig(fig, stem):
    # Each figure is saved as .png and .pdf in FIG_DIR
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{opsim_name}_{stem}.{ext}"), dpi=150, bbox_inches="tight")


def grid_axes(n, ncols=3, sharex=True):
    # Grid of n panels (3 columns); the extra panels are hidden
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.5 * ncols, 4.2 * nrows), sharex=sharex, squeeze=False)
    axes = axes.ravel()
    for ax in axes[n:]:
        ax.axis("off")
    return fig, axes


def label_grid(axes, n, ncols=3):
    # Axis labels: time at the bottom, fraction on the left
    for k, ax in enumerate(axes[:n]):
        if k % ncols == 0:
            ax.set_ylabel("Fraction of visits (%)")
        if k + ncols >= n:
            ax.set_xlabel("Time since survey start (years)")
        ax.grid(alpha=0.3)


def band_legend(fig, ax):
    # Filter legend placed above the panels, outside the axes, so that it does not shrink them
    handles, names = ax.get_legend_handles_labels()
    fig.legend(
        handles, names, ncol=len(BANDS), loc="upper center", bbox_to_anchor=(0.5, 0.955), title="filter"
    )

## 4. Monthly fractions

### Figure 1 - Monthly filter balance as stacked bars (`<run>_stacked_fractions.png`)

One panel per category. Each bar is a month; its coloured segments give the fraction of each filter (sum = 100%). If the balance is stable, the colour bands stay horizontal. A month without a bar has fewer than `MIN_VISITS_MONTH` visits in the category. The total number of visits of the category is in the panel title.

In [ ]:
fig1, axes = grid_axes(len(CATEGORIES))
for ax, cat in zip(axes, CATEGORIES):
    bottom = np.zeros(N_MONTHS)
    for i, b in enumerate(BANDS):
        h = np.nan_to_num(100 * frac[cat][:, i])
        ax.bar(x, h, width=1 / 12, bottom=bottom, color=BAND_COLORS[b], edgecolor="none", label=b)
        bottom += h
    ax.set_title(f"{cat} ({counts[cat].sum()})")
    ax.set_xlim(0, N_YEARS)
    ax.set_ylim(0, 100)
label_grid(axes, len(CATEGORIES))
band_legend(fig1, axes[0])
fig1.suptitle(f"Monthly filter balance - {opsim_name}")
fig1.tight_layout(rect=(0, 0, 1, 0.92))
save_fig(fig1, "stacked_fractions")

### Figure 2 - Fraction of each filter versus time (`<run>_fractions_vs_time.png`)

One panel per category, one curve per filter (solid line: fraction of the month). The dashed line is the **mean fraction** of the filter, and the coloured band is the **expected statistical fluctuation** (± 1 binomial sigma) around this mean. If the balance is constant in time, the curve of each filter stays inside its band, with about 68% of the months inside.

Caution: the visits are not independent (pairs of visits, blobs in one filter), so the real dispersion is somewhat larger than the binomial band even for a perfectly stationary strategy. A systematic drift or a yearly modulation (seasons) cannot come from this effect, however.

In [ ]:
fig2, axes = grid_axes(len(CATEGORIES))
for ax, cat in zip(axes, CATEGORIES):
    for i, b in enumerate(BANDS):
        fm = f_mean[cat][i]
        ax.fill_between(
            x,
            100 * (fm - sigma[cat][:, i]),
            100 * (fm + sigma[cat][:, i]),
            color=BAND_COLORS[b],
            alpha=0.5,
            lw=0,
        )
        ax.axhline(100 * fm, color=BAND_COLORS[b], lw=1.0, ls="--")
        ax.plot(x, 100 * frac[cat][:, i], color=BAND_COLORS[b], lw=2, label=b)
    ax.set_title(f"{cat} ({counts[cat].sum()})")
    ax.set_xlim(0, N_YEARS)
    ax.set_ylim(bottom=0)
label_grid(axes, len(CATEGORIES))
band_legend(fig2, axes[0])
fig2.suptitle(f"Fraction of each filter versus time - {opsim_name}")
fig2.tight_layout(rect=(0, 0, 1, 0.92))
save_fig(fig2, "fractions_vs_time")

### Figure 3 - Filter balance compared between survey types (`<run>_balance_by_survey.png`)

Fraction of visits in each filter over the 10 years, for each category (one colour per category, grouped by filter). This is the figure that answers the question: is the balance different from one survey type to another? The black diamonds give the **indicative** reference balance of the WFD (`REF_WFD_VISITS` of the configuration, to be checked for v5.3), to be compared with the `WFD` bar. The `All visits` category gives the overall balance of the survey.

In [ ]:
fig3, ax = plt.subplots(figsize=(14, 6))
n_cat = len(CATEGORIES)
width = 0.85 / n_cat
tab = plt.get_cmap("tab20")
for k, cat in enumerate(CATEGORIES):
    f10 = 100 * counts[cat].sum(axis=0) / counts[cat].sum()
    pos = np.arange(len(BANDS)) + (k - (n_cat - 1) / 2) * width
    ax.bar(pos, f10, width=width * 0.95, color=tab(k % 20), label=cat)

# Indicative reference balance of the WFD
ref = np.array([REF_WFD_VISITS[b] for b in BANDS], dtype=float)
ax.plot(np.arange(len(BANDS)), 100 * ref / ref.sum(), "kD", ms=9, label="WFD: indicative reference")

ax.set_xticks(np.arange(len(BANDS)))
ax.set_xticklabels(BANDS)
ax.set_xlabel("Filter")
ax.set_ylabel("Fraction of visits over 10 years (%)")
ax.grid(alpha=0.3, axis="y")
ax.legend(ncol=2, fontsize=11, loc="upper left")
ax.set_title(f"Filter balance per survey type - {opsim_name}")
fig3.tight_layout()
save_fig(fig3, "balance_by_survey")

### Figure 4 - Filter balance of each survey type, as bar plots (`<run>_fractions_per_survey_bars.png`)

One panel per category. Each vertical bar is the **fraction (%) of the visits of the category in one filter** over the 10 years, with the filters in the order u, g, r, i, z, y (bar colours = filter colours). The number written above each bar is the **total number of visits in this filter**. The total number of visits of the category is in the panel title. Unlike Figure 3, every category has its own vertical scale, so the shape of the balance of a small category (a mini-survey, a micro-survey) is as readable as that of the WFD.

In [ ]:
fig_bars, axes = grid_axes(len(CATEGORIES), sharex=False)
for ax, cat in zip(axes, CATEGORIES):
    n_band = counts[cat].sum(axis=0)  # number of visits in each filter, order u, g, r, i, z, y
    f_band = 100 * n_band / n_band.sum()
    ax.bar(range(len(BANDS)), f_band, color=[BAND_COLORS[b] for b in BANDS], edgecolor="k", linewidth=0.6)
    # Total number of visits in each filter, written above the bar
    for i in range(len(BANDS)):
        ax.annotate(
            f"{n_band[i]:,}",
            (i, f_band[i]),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=10,
        )
    ax.set_ylim(0, 1.18 * f_band.max())  # room for the text above the highest bar
    ax.set_xticks(range(len(BANDS)))
    ax.set_xticklabels(BANDS)
    ax.set_title(f"{cat} ({n_band.sum():,})")
    ax.grid(alpha=0.3, axis="y")
for k, ax in enumerate(axes[: len(CATEGORIES)]):
    if k % 3 == 0:
        ax.set_ylabel("Fraction of visits (%)")
    if k + 3 >= len(CATEGORIES):
        ax.set_xlabel("Filter")
fig_bars.suptitle(f"Filter balance of each survey type - {opsim_name}")
fig_bars.tight_layout(rect=(0, 0, 1, 0.96))
save_fig(fig_bars, "fractions_per_survey_bars")

## 5. Time stability test

For each category and each filter, three diagnostics on the usable months:

1. **RMS of the pulls**: `pull = (f_b(month) - f_mean) / sigma_binomial`. For a constant fraction and independent visits, the RMS is 1. A value clearly above 1 means a variation larger than the statistics (real effect, or correlated visits).
2. **Drift over 10 years**: slope of a weighted linear fit of `f_b` versus time, multiplied by 10 years, in **percentage points** (pp). It is flagged `*` when it exceeds 3 times its error.
3. **Chi-square of homogeneity** of the (month x filter) table: tests globally, for a category, whether the filter balance is the same every month. `chi2/dof` close to 1: stable; much larger than 1: the balance varies. Since the visits are correlated, `chi2/dof` is biased upward: look mostly at the **relative differences between categories** and at the **drift**, rather than at the absolute p-value.

The results are saved in `DATA10_FilterBalance/` (`..._stability_per_band.csv` and `..._stability_per_category.csv`).

In [ ]:
n_c, n_b = len(CATEGORIES), len(BANDS)
pull_rms = np.full((n_c, n_b), np.nan)
drift = np.full((n_c, n_b), np.nan)
drift_err = np.full((n_c, n_b), np.nan)
chi2_rows = {}

for ic, cat in enumerate(CATEGORIES):
    g = good[cat]
    c = counts[cat][g]
    nm = n_month[cat][g]
    fm = f_mean[cat]
    tm = x[g]
    for ib in range(n_b):
        if fm[ib] <= 0 or fm[ib] >= 1:
            continue
        fb = c[:, ib] / nm
        sig = np.sqrt(fm[ib] * (1 - fm[ib]) / nm)
        pull_rms[ic, ib] = np.sqrt(np.mean(((fb - fm[ib]) / sig) ** 2))
        # Weighted linear fit f_b(t) = a t + b; drift = a * N_YEARS (in percentage points)
        p, cov = np.polyfit(tm, fb, 1, w=1 / sig, cov="unscaled")
        drift[ic, ib] = 100 * p[0] * N_YEARS
        drift_err[ic, ib] = 100 * np.sqrt(cov[0, 0]) * N_YEARS

    # Chi-square of homogeneity of the (month x filter) table, filters with zero fraction excluded
    used = fm > 0
    expected = nm[:, None] * fm[None, :]
    chi2 = (((c - expected) ** 2) / np.where(expected > 0, expected, 1))[:, used].sum()
    dof = (g.sum() - 1) * (used.sum() - 1)
    chi2_rows[cat] = {
        "N months": int(g.sum()),
        "chi2": chi2,
        "dof": int(dof),
        "chi2/dof": chi2 / dof,
        "p-value": stats.chi2.sf(chi2, dof),
    }

# Tables: per filter and per category
per_band = pd.concat(
    {
        "pull RMS": pd.DataFrame(pull_rms, index=CATEGORIES, columns=BANDS),
        "drift 10 yr (pp)": pd.DataFrame(drift, index=CATEGORIES, columns=BANDS),
        "drift error (pp)": pd.DataFrame(drift_err, index=CATEGORIES, columns=BANDS),
    },
    axis=1,
)
per_band.round(3).to_csv(os.path.join(DATA_DIR, f"{opsim_name}_stability_per_band.csv"))
per_cat = pd.DataFrame(chi2_rows).T
per_cat.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_stability_per_category.csv"))
per_cat.style.format({"chi2": "{:.1f}", "chi2/dof": "{:.2f}", "p-value": "{:.2e}"})

### Figure 5 - Time stability per category and per filter (`<run>_stability.png`)

Two maps (categories in rows, filters in columns):

- **left**: RMS of the pulls (1 = purely statistical fluctuation);
- **right**: linear drift of the fraction over 10 years, in percentage points (`*`: significant at more than 3 sigma). A drift of a few pp on a filter at 20% is a real effect of the strategy; a drift of 0.1 pp is not one from the point of view of the science.

In [ ]:
fig4, (axl, axr) = plt.subplots(1, 2, figsize=(16, 0.55 * n_c + 3.5))

# Map of the RMS of the pulls
im = axl.imshow(pull_rms, cmap="viridis", vmin=0, aspect="auto")
axl.set_title("RMS of the pulls (1 = pure statistics)")
fig4.colorbar(im, ax=axl, fraction=0.04, pad=0.02)
for ic in range(n_c):
    for ib in range(n_b):
        if np.isfinite(pull_rms[ic, ib]):
            axl.text(ib, ic, f"{pull_rms[ic, ib]:.1f}", ha="center", va="center", color="w", fontsize=12)

# Map of the drifts (symmetric scale)
vmax = np.nanmax(np.abs(drift))
im = axr.imshow(drift, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
axr.set_title("Drift over 10 years (percentage points)")
fig4.colorbar(im, ax=axr, fraction=0.04, pad=0.02)
for ic in range(n_c):
    for ib in range(n_b):
        if np.isfinite(drift[ic, ib]):
            star = "*" if abs(drift[ic, ib]) > 3 * drift_err[ic, ib] else ""
            axr.text(ib, ic, f"{drift[ic, ib]:+.1f}{star}", ha="center", va="center", fontsize=12)

for ax in (axl, axr):
    ax.set_xticks(range(n_b))
    ax.set_xticklabels(BANDS)
    ax.set_xlabel("Filter")
axl.set_yticks(range(n_c))
axl.set_yticklabels(CATEGORIES)
axr.set_yticks([])
fig4.suptitle(f"Time stability of the filter balance - {opsim_name}")
fig4.tight_layout()
save_fig(fig4, "stability")

## 6. Reading the results

- **Difference between survey types**: Figure 3 and the table of section 3. If the bars of `WFD`, `DDF` and the mini-surveys differ, the balance does depend on the survey type, as expected.
- **No evolution in time**: Figures 1, 2 and 5. For each category, expect flat curves (Figure 2), a pull RMS close to 1 (above it, check whether the excess comes from correlated visits or from a real structure) and drifts compatible with zero (Figure 5).
- **Points of attention**: (1) a modulation with a 1-year period in Figure 2 indicates a seasonal effect (visibility of the regions); (2) categories with low statistics (mini and micro-surveys) have wide statistical bands, hence low-sensitivity tests; (3) the ramp-up of the rolling cadence should not change the filter balance of the total WFD, but can change that of sub-regions.
- **Possible extensions**: fractions in exposure time instead of number of visits; balance per individual DDF (see `04_DDF_NVisitsVsTime`); comparison with another run (change `opsdb`).